# Mamografías benigna/maligna — Grupo 8 (Colab)
**Prototipo académico; no apto para diagnóstico ni decisiones clínicas.**



In [3]:
import subprocess;print(subprocess.run('cd /content && rm -f parte_* && split -b 8m -d paquete_entrega.zip parte_ && mkdir -p /content/drive/MyDrive/mamografias_g8/paquete_partes && rm -f /content/drive/MyDrive/mamografias_g8/paquete_partes/* && cp parte_* /content/drive/MyDrive/mamografias_g8/paquete_partes/ && ls -l parte_* && md5sum paquete_entrega.zip',shell=True,capture_output=True,text=True).stdout)

-rw-r--r-- 1 root root 8388608 Nov 25 22:15 parte_00
-rw-r--r-- 1 root root 5546056 Nov 25 22:15 parte_01
0423dd800087d6d0d42fe8b8d6114f58  paquete_entrega.zip



In [ ]:
!nvidia-smi

## 1. Código del repositorio


In [ ]:
!git clone https://github.com/JoseG-SotoMendez/mamografias_g8.git
%cd /content/mamografias_g8
!git log -1 --format='%h %ad %an %s' --date=iso
!pip install -q pydicom gradio tabulate kagglehub

## 2. Google Drive (para no perder resultados si Colab se desconecta)
`DRIVE` es la carpeta donde se guardan `runs/`, `reports/` y la partición. Si la sesión se cae, la celda siguiente restaura lo que ya estaba guardado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/mamografias_g8'
!mkdir -p {DRIVE}
# Restaurar corridas guardadas antes (si existen)
!cp -rn {DRIVE}/runs {DRIVE}/reports . 2>/dev/null; ls runs reports

## 3. Descargar CBIS-DDSM (versión JPEG de Kaggle)


In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download('awsaf49/cbis-ddsm-breast-cancer-image-dataset')
print(DATA_ROOT)
!ls {DATA_ROOT}

## 4. Preparación y partición por paciente
Decisiones del equipo: `--lesion` (mass/calc/both) y `--bwc` (benign/exclude). Documentarlas en la bitácora.

In [ ]:
!python src/prepare_data.py --source kaggle --root {DATA_ROOT} --lesion both --bwc benign --seed 42
!python src/check_leakage.py

In [ ]:
# Guardar la partición en Drive: es la evidencia del split que va al repositorio
!mkdir -p {DRIVE}/splits && cp data/splits/lesiones.csv data/splits/resumen.json {DRIVE}/splits/
!cat data/splits/resumen.json

In [ ]:
!python src/eda.py
from IPython.display import Image, display
for f in ['balance_clases','tipo_vs_clase','tamanos','ejemplos_benignas','ejemplos_malignas']:
    display(Image(f'reports/eda/{f}.png'))

## 5. Entrenamiento
Prueba rápida primero (1 época, 200 lesiones) para verificar que todo corre.

In [ ]:
!python src/train.py --name prueba --model resnet18 --epochs 1 --max-train 200 --aug

In [ ]:
# Línea base
!python src/train.py --name R0_smallcnn --model smallcnn --epochs 25 --lr 1e-3 --aug

In [ ]:
# Transferencia de aprendizaje
!python src/train.py --name R1_resnet18_aug --model resnet18 --epochs 15 --lr 1e-4 --aug

In [ ]:
# Experimento controlado: igual que R1 pero SIN aumentos (una sola decisión cambia)
!python src/train.py --name R2_resnet18_noaug --model resnet18 --epochs 15 --lr 1e-4

In [ ]:
for r in ['R0_smallcnn','R1_resnet18_aug','R2_resnet18_noaug']:
    display(Image(f'runs/{r}/curvas.png'))

## 6. Evaluación en VALIDACIÓN (aquí se elige el umbral)

In [ ]:
for r in ['R0_smallcnn','R1_resnet18_aug','R2_resnet18_noaug']:
    !python src/evaluate.py --run runs/{r} --split val --strategy sens --target-sens 0.90
!python src/compare_runs.py --split val

In [ ]:
R = 'R1_resnet18_aug'  # cambiar por la corrida elegida en validación
for f in ['matriz_confusion','roc','umbral_vs_metricas','falsos_negativos','falsos_positivos']:
    p = f'runs/{R}/eval_val/{f}.png'
    import os
    if os.path.exists(p): display(Image(p))

## 7. Prueba reservada — ejecutar UNA sola vez al final
Con el modelo y la estrategia de umbral ya decididos en validación.

In [ ]:
GANADOR = 'R1_resnet18_aug'  # decisión del equipo
# !python src/evaluate.py --run runs/{GANADOR} --split test --strategy sens --target-sens 0.90
# !python src/compare_runs.py --split test

## 8. Interfaz Gradio

In [ ]:
!python app.py --run runs/{GANADOR} --share

## 9. Guardar todo en Drive
Correr al final de cada sesión (y después de cada entrenamiento largo). Los pesos `best.pt` se guardan en Drive, **no** en GitHub.

In [ ]:
!cp -r runs reports {DRIVE}/
!ls -R {DRIVE} | head -50

## 10. Indicaciones del profesor (revisión del día 7)
1. **Más épocas:** R3 = R1 con hasta 30 épocas y paciencia 7 (validación decide la mejor época).
2. **Más datos de entrenamiento:** R4 = misma configuración entrenada con **train + val** durante las épocas que eligió R3. No usa prueba.
3. **Imágenes totalmente nuevas:** mini-MIAS (Reino Unido, otro escáner): recortes de lesión evaluados con R3 y R4, umbral fijado antes en validación de CBIS-DDSM.

In [ ]:
!python src/train.py --name R3_resnet18_aug_30ep --model resnet18 --epochs 30 --patience 7 --lr 1e-4 --aug
!python src/evaluate.py --run runs/R3_resnet18_aug_30ep --split val --strategy sens --target-sens 0.90
!python src/compare_runs.py --split val

In [ ]:
import json
E = json.load(open('runs/R3_resnet18_aug_30ep/config.json'))['mejor_epoca']
print('Épocas elegidas en validación:', E)
!python src/train.py --name R4_final_trainval --model resnet18 --epochs {E} --lr 1e-4 --aug --incluir-val

In [ ]:
import kagglehub
MIAS = kagglehub.dataset_download('kmader/mias-mammography')
!python src/prepare_mias.py --root {MIAS} --out data/externo/mias

In [ ]:
for r in ['R1_resnet18_aug', 'R3_resnet18_aug_30ep']:
    !python src/evaluate_external.py --run runs/{r} --csv data/externo/mias/lesiones_mias.csv
!python src/evaluate_external.py --run runs/R4_final_trainval --csv data/externo/mias/lesiones_mias.csv --threshold-from runs/R3_resnet18_aug_30ep/threshold.json
!cp -r runs reports data/externo {DRIVE}/